In [ ]:
import pytest
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.exc import IntegrityError
from your_project.database import Base  # Adapte à ton database.py
from your_project.models import *      # Importe tes modèles (User, Game...)

@pytest.fixture(scope="function")
def db_session():
    engine = create_engine(
        "sqlite:///:memory:", 
        connect_args={"check_same_thread": False}
    )
    Base.metadata.create_all(engine)
    
    Session = sessionmaker(bind=engine)
    session = Session()
    
    yield session  # Les tests utilisent cette session
    
    session.rollback()
    session.close()
    Base.metadata.drop_all(engine)


In [ ]:
# crud.py (ton code existant)
def create_user(session, name, email):
    user = User(name=name, email=email)
    session.add(user)
    session.commit()
    return user

def get_user(session, user_id):
    return session.get(User, user_id)

def update_user(session, user_id, new_name):
    user = session.get(User, user_id)
    if user:
        user.name = new_name
        session.commit()
    return user

def delete_user(session, user_id):
    user = session.get(User, user_id)
    if user:
        session.delete(user)
        session.commit()


In [ ]:
from your_project.crud import create_user, get_user, update_user, delete_user
from your_project.models import User

def test_create_and_get_user(db_session):
    user = create_user(db_session, "Alice", "alice@test.com")
    assert user.name == "Alice"
    
    fetched = get_user(db_session, user.id)
    assert fetched.email == "alice@test.com"

def test_update_user(db_session):
    user = create_user(db_session, "Bob", "bob@test.com")
    updated = update_user(db_session, user.id, "Bobby")
    assert updated.name == "Bobby"

def test_delete_user(db_session):
    user = create_user(db_session, "Charlie", "charlie@test.com")
    delete_user(db_session, user.id)
    assert get_user(db_session, user.id) is None

def test_unique_constraint(db_session):
    create_user(db_session, "Dup", "dup@test.com")
    user2 = User(name="Dup", email="other@test.com")
    db_session.add(user2)
    with pytest.raises(IntegrityError):
        db_session.commit()
    db_session.rollback()


In [1]:
import gradio as gr
import time

def traitement_long(nom):
    # Simulation d'un traitement
    time.sleep(1)
    
    if not nom:
        # Affiche un toast d'avertissement (jaune/orange)
        gr.Warning("Veuillez entrer un nom avant de cliquer !")
        return "Erreur"
    
    # Affiche un toast d'information (bleu/vert)
    gr.Info(f"Traitement terminé pour {nom} !")
    return f"Bonjour {nom}"

with gr.Blocks() as demo:
    gr.Markdown("# Exemple de Toast Gradio")
    
    input_text = gr.Textbox(label="Votre nom")
    btn = gr.Button("Lancer l'action")
    output_text = gr.Textbox(label="Résultat")
    
    # L'appel à gr.Info/Warning se fait directement à l'intérieur de la fonction
    btn.click(fn=traitement_long, inputs=input_text, outputs=output_text)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Affichage d'un bouton pour saisir du texte et l'ajouté à une liste 
# Si le texte saisi existe deja dans la liste un demande de confirmation d'ajout est affiché
# Si oui ajout 
# Si non pas d'ajout 

import gradio as gr

def verifier_et_ajouter(nouveau_nom, liste_actuelle):
    # Nettoyage de la saisie
    nouveau_nom = nouveau_nom.strip()
    if not nouveau_nom:
        return gr.update(), gr.update(), "⚠️ Veuillez saisir un nom."

    # Si le nom existe déjà : on affiche la zone de confirmation
    if nouveau_nom in liste_actuelle:
        return gr.update(visible=True), gr.update(), f"❓ '{nouveau_nom}' est déjà dans la liste. Voulez-vous l'ajouter quand même ?"
    
    # Si le nom est nouveau : on l'ajoute directement
    liste_actuelle.append(nouveau_nom)
    return gr.update(visible=False), liste_actuelle, f"✅ '{nouveau_nom}' ajouté à la liste."

def confirmation_forcee(nouveau_nom, liste_actuelle):
    # L'utilisateur a cliqué sur "Oui, ajouter quand même"
    liste_actuelle.append(nouveau_nom)
    return gr.update(visible=False), liste_actuelle, f"✅ '{nouveau_nom}' ajouté (doublon accepté).", ""

with gr.Blocks() as demo:
    # 1. État interne pour stocker la liste (State)
    noms_state = gr.State(["pierre", "paul", "jack"])
    
    gr.Markdown("### Gestion de liste avec contrôle de doublons")
    
    # 2. Interface de saisie
    with gr.Row():
        input_nom = gr.Textbox(label="Nouveau prénom", placeholder="Entrez un nom...")
        btn_ajouter = gr.Button("Ajouter à la liste", variant="primary")
    
    # 3. Zone de confirmation (cachée par défaut)
    with gr.Column(visible=False, variant="panel") as zone_confirm_doublon:
        message_alerte = gr.Markdown()
        with gr.Row():
            btn_annuler = gr.Button("Annuler")
            btn_oui_doublon = gr.Button("Oui, ajouter quand même", variant="secondary")
    
    # 4. Affichage du résultat et de la liste
    resultat_info = gr.Markdown()
    affichage_liste = gr.Textbox(label="Contenu de la liste", value="pierre, paul, jack", interactive=False)

    # --- LOGIQUE ---

    # Action du bouton principal
    btn_ajouter.click(
        fn=verifier_et_ajouter,
        inputs=[input_nom, noms_state],
        outputs=[zone_confirm_doublon, noms_state, message_alerte]
    ).then( # Mise à jour de la vue texte de la liste
        fn=lambda x: ", ".join(x), inputs=noms_state, outputs=affichage_liste
    )

    # Action du bouton "Annuler" dans la zone de confirmation
    btn_annuler.click(lambda: gr.update(visible=False), None, zone_confirm_doublon)

    # Action du bouton "Oui, ajouter quand même"
    btn_oui_doublon.click(
        fn=confirmation_forcee,
        inputs=[input_nom, noms_state],
        outputs=[zone_confirm_doublon, noms_state, resultat_info, input_nom]
    ).then(
        fn=lambda x: ", ".join(x), inputs=noms_state, outputs=affichage_liste
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [8]:
import gradio as gr
import pandas as pd

# Votre exemple de données
donnees_initiales = [
    {'id': 1, 'resume_name': 'test', 'resume': 'NULL_TEXT_EMPTY'},
    {'id': 2, 'resume_name': 'test2', 'resume': 'testresume'}
]

def afficher_donnees():
    # Transformation de la liste de dictionnaires en DataFrame Pandas
    df = pd.DataFrame(donnees_initiales)
    return df

with gr.Blocks() as demo:
    gr.Markdown("### Affichage de la liste de CV (Dataframe)")
    
    # On définit le composant Dataframe
    # interactive=False le rend uniquement consultable
    tableau = gr.Dataframe(
        value=afficher_donnees(), # Charge les données au lancement
        interactive=False,
        label="Liste des résumés"
    )
    
    btn_refresh = gr.Button("Rafraîchir les données")
    btn_refresh.click(fn=afficher_donnees, outputs=tableau)

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [9]:
import gradio as gr
import pandas as pd

# 1. Préparation des données
initial_data = [
    {'id': 1, 'resume_name': 'CV_Pierre', 'resume': 'Expérience en Python...'},
    {'id': 2, 'resume_name': 'CV_Paul', 'resume': 'Expert en Data Science...'},
    {'id': 3, 'resume_name': 'CV_Jack', 'resume': 'Développeur Fullstack...'}
]
df_global = pd.DataFrame(initial_data)

# 2. Fonctions logiques
def gerer_clic(evt: gr.SelectData):
    """Déclenché lors d'un clic sur une cellule"""
    index = evt.index[0]
    nom_cv = df_global.iloc[index]['resume_name']
    
    # On retourne la zone de confirmation visible avec les infos de la ligne
    return (
        gr.update(visible=True), 
        f"### 🗑️ Action sur : {nom_cv}\nVoulez-vous supprimer ce résumé ?",
        index
    )

def supprimer_et_fermer(index_ligne):
    global df_global
    # Suppression dans le DataFrame
    df_global = df_global.drop(df_global.index[index_ligne]).reset_index(drop=True)
    
    # Toast de confirmation (si votre version le supporte)
    try: gr.Info("Supprimé !")
    except: pass
    
    return df_global, gr.update(visible=False)

# 3. Interface
with gr.Blocks() as demo:
    gr.Markdown("## 📑 Gestion des résumés")
    
    # Variable cachée pour stocker l'index sélectionné
    selected_index = gr.State(None)
    
    # Affichage du tableau (interactive=False évite l'édition de texte accidentelle)
    tableau = gr.Dataframe(
        value=df_global, 
        interactive=False,
        label="Cliquez sur une ligne pour la supprimer"
    )

    # Zone de confirmation qui apparaît sous le tableau
    with gr.Column(visible=False, variant="panel") as zone_confirmation:
        message = gr.Markdown()
        with gr.Row():
            btn_non = gr.Button("Annuler")
            btn_oui = gr.Button("Confirmer la suppression", variant="stop")

    # --- ÉVÉNEMENTS ---
    
    # Au clic sur une cellule du tableau
    tableau.select(
        fn=gerer_clic, 
        outputs=[zone_confirmation, message, selected_index]
    )
    
    # Boutons de la zone de confirmation
    btn_non.click(lambda: gr.update(visible=False), None, zone_confirmation)
    
    btn_oui.click(
        fn=supprimer_et_fermer,
        inputs=[selected_index],
        outputs=[tableau, zone_confirmation]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [19]:
import gradio as gr
import pandas as pd

# Données exemples
initial_data = [
    {'id': 1, 'resume_name': 'CV_Pierre', 'resume': 'Détails de Pierre...'},
    {'id': 2, 'resume_name': 'CV_Paul', 'resume': 'Détails de Paul...'}
]
df_global = pd.DataFrame(initial_data)

def gerer_clic(evt: gr.SelectData):
    index = evt.index[0]
    resume_complet = df_global.iloc[index]['resume']
    # On affiche le contenu ET on rend la zone de suppression visible
    return resume_complet, gr.update(visible=True), index

def supprimer_ligne(index):
    global df_global
    df_global = df_global.drop(df_global.index[index]).reset_index(drop=True)
    return df_global, "", gr.update(visible=False)

with gr.Blocks() as demo:
    idx_stock = gr.State(None)
    
    with gr.Row():
        # Gauche : Le Tableau
        with gr.Column(scale=2):
            tableau = gr.Dataframe(value=df_global, interactive=False)
        
        # Droite : L'affichage dynamique
        with gr.Column(scale=3):
            display_area = gr.Textbox(label="Aperçu du résumé", lines=10)
            
            # Cette zone n'apparaît que quand une ligne est cliquée
            with gr.Row(visible=False) as zone_action:
                gr.Markdown("⚠️ *Action rapide :*")
                btn_del = gr.Button("🗑️ Supprimer ce profil", variant="stop", size="sm")

    # Logique
    tableau.select(fn=gerer_clic, outputs=[display_area, zone_action, idx_stock])
    btn_del.click(fn=supprimer_ligne, inputs=idx_stock, outputs=[tableau, display_area, zone_action])

demo.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [18]:
import gradio as gr
import pandas as pd

# 1. Données simulées
data = [
    {'id': 1, 'resume_name': 'CV_Pierre', 'resume': 'Détails de Pierre...'},
    {'id': 2, 'resume_name': 'CV_Paul', 'resume': 'Détails de Paul...'},
    {'id': 3, 'resume_name': 'CV_Jack', 'resume': 'Détails de Jack...'}
]
df_global = pd.DataFrame(data)

# 2. Fonctions de logique
def mettre_a_jour_donnees(nouveau_df):
    """
    Cette fonction se lance à chaque modification du tableau
    (Clic droit > Delete Row ou édition de cellule).
    """
    global df_global
    df_global = nouveau_df
    # On affiche un petit toast pour confirmer la synchro
    gr.Info("Données synchronisées avec la variable Python")
    return nouveau_df

def afficher_contenu(evt: gr.SelectData):
    """Affiche le résumé lors du simple clic gauche."""
    index = evt.index[0]
    # On récupère le contenu depuis le DataFrame global mis à jour
    try:
        nom = df_global.iloc[index]['resume_name']
        contenu = df_global.iloc[index]['resume']
        return f"### 📄 Résumé de {nom}\n\n{contenu}"
    except IndexError:
        return "Sélection invalide"

with gr.Blocks() as demo:
    gr.Markdown("# 📁 Gestionnaire avec Synchro Automatique")
    gr.Markdown("💡 *Clic gauche pour lire | Clic droit sur une ligne pour supprimer*")
    
    with gr.Row():
        with gr.Column(scale=2):
            # interactive=True est indispensable pour le clic droit
            table = gr.Dataframe(
                value=df_global,
                interactive=True,
                type="pandas"
            )
        
        with gr.Column(scale=1):
            viewer = gr.Markdown("Sélectionnez un candidat...")

    # --- ÉVÉNEMENTS ---

    # 1. Clic gauche pour l'affichage
    table.select(fn=afficher_contenu, outputs=viewer)

    # 2. Synchronisation automatique lors du clic droit (Delete Row)
    # Dès que le tableau change, on met à jour la variable globale.
    table.change(fn=mettre_a_jour_donnees, inputs=table, outputs=table)
    
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


In [20]:
import gradio as gr
import pandas as pd

# Données exemples
initial_data = [
    {'id': 1, 'resume_name': 'CV_Pierre', 'resume': 'Détails de Pierre...'},
    {'id': 2, 'resume_name': 'CV_Paul', 'resume': 'Détails de Paul...'}
]
df_global = pd.DataFrame(initial_data)

def gerer_clic(evt: gr.SelectData):
    index = evt.index[0]
    resume_complet = df_global.iloc[index]['resume']
    # On affiche le contenu, on montre le bouton supprimer initial, on cache la confirmation
    return resume_complet, gr.update(visible=True), gr.update(visible=False), index

def demander_confirmation():
    # Cache le bouton "Supprimer" et affiche les boutons "Oui/Non"
    return gr.update(visible=False), gr.update(visible=True)

def annuler_suppression():
    # Revient à l'état initial
    return gr.update(visible=True), gr.update(visible=False)

def supprimer_ligne(index):
    global df_global
    # Suppression réelle dans le DataFrame
    df_global = df_global.drop(df_global.index[index]).reset_index(drop=True)
    # On réinitialise tout l'affichage
    return df_global, "", gr.update(visible=False), gr.update(visible=False)

with gr.Blocks() as demo:
    idx_stock = gr.State(None)
    
    with gr.Row():
        # Gauche : Le Tableau
        with gr.Column(scale=2):
            gr.Markdown("### 📋 Liste des CV")
            tableau = gr.Dataframe(value=df_global, interactive=False)
        
        # Droite : L'affichage dynamique
        with gr.Column(scale=3):
            display_area = gr.Textbox(label="Aperçu du résumé", lines=10)
            
            # ÉTAPE 1 : Bouton de suppression initial
            btn_pre_del = gr.Button("🗑️ Supprimer ce profil", variant="stop", visible=False)
            
            # ÉTAPE 2 : Zone de confirmation (cachée par défaut)
            with gr.Row(visible=False, variant="panel") as zone_confirm:
                gr.Markdown("❗ **Confirmer la suppression ?**")
                btn_non = gr.Button("Annuler", size="sm")
                btn_oui = gr.Button("Oui, Supprimer définitivement", variant="stop", size="sm")

    # --- LOGIQUE ---

    # 1. Clic sur le tableau -> Affiche le texte et le premier bouton de suppression
    tableau.select(
        fn=gerer_clic, 
        outputs=[display_area, btn_pre_del, zone_confirm, idx_stock]
    )
    
    # 2. Clic sur "Supprimer" -> Affiche la validation Oui/Non
    btn_pre_del.click(
        fn=demander_confirmation, 
        outputs=[btn_pre_del, zone_confirm]
    )

    # 3. Clic sur "Annuler" -> Retour au bouton simple
    btn_non.click(
        fn=annuler_suppression, 
        outputs=[btn_pre_del, zone_confirm]
    )

    # 4. Clic sur "Oui" -> Suppression effective et nettoyage
    btn_oui.click(
        fn=supprimer_ligne, 
        inputs=idx_stock, 
        outputs=[tableau, display_area, btn_pre_del, zone_confirm]
    )

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
